### SEED GATHERING GET CONTENT

In [1]:
!git clone https://github.com/tree-sitter/tree-sitter-typescript
!cd tree-sitter-typescript/typescript && npm install

Cloning into 'tree-sitter-typescript'...
remote: Enumerating objects: 5205, done.
remote: Counting objects: 100% (1629/1629), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 5205 (delta 1537), reused 1468 (delta 1468), pack-reused 3576 (from 4)
Receiving objects: 100% (5205/5205), 142.10 MiB | 20.70 MiB/s, done.
Resolving deltas: 100% (3261/3261), done.
⠙
up to date, audited 1 package in 614ms
⠙
found 0 vulnerabilities
⠙

In [2]:
!git clone https://github.com/kamaravichow/seed-gathering.git

Cloning into 'seed-gathering'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 28 (delta 9), reused 3 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 21.30 KiB | 21.30 MiB/s, done.
Resolving deltas: 100% (9/9), done.


In [3]:
!cp -r seed-gathering/* /content/
!rm -rf seed-gathering

In [4]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 575.6/575.6 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.4/326.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
!pip install tree_sitter==0.20.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 5.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for tree_sitter: filename=tree_sitter-0.20.1-cp311-cp311-linux_x86_64.whl size=425848 sha256=f516204567e48347a8f8b22a0c5363626e88aa4f0fdcb0ff109073a5c0551acf
  Stored in directory: /root/.cache/pip/wheels/6c/be/3a/6841c52111041475b3c693b347ac03f305330cbf7fb1c6365d
Successfully built tree_sitter
  Attempting uninstall: tree_sitter
    Found existing installation: tree-sitter 0.24.0
    Uninstalling tree-sitter-0.24.0:
      Successfully uninstalled tree-sitter-0.24.0


In [6]:
from tree_sitter import Language, Parser

# Build once (outside notebook ideally)
Language.build_library(
  'build/my-languages.so',
  ['tree-sitter-typescript/typescript']
)

TS_LANGUAGE = Language('build/my-languages.so', 'typescript')

def make_parser():
    parser = Parser()
    parser.set_language(TS_LANGUAGE)
    return parser

In [7]:
TS_LANGUAGE = Language('build/my-languages.so', 'typescript')

In [8]:
def node_to_string(source_bytes, node):
    return source_bytes[node.start_byte:node.end_byte].decode("utf8")

# Tree-sitter TypeScript function query
TS_FUNCTION_QUERY = TS_LANGUAGE.query("""
(function_declaration
  name: (identifier) @function.name
  body: (statement_block) @function.body) @function.def
""")

def extract_ts_functions(source_code):
    try:
        buf = bytes(source_code, "utf8")
        parser = make_parser()
        tree = parser.parse(buf)
        captures = TS_FUNCTION_QUERY.captures(tree.root_node)
        return [node_to_string(buf, node) for node, typ in captures if typ == "function.def"]
    except Exception as e:
        print("Parse error:", e)
        return []

In [9]:
!pip install boto3 botocore smart_open

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 9.1 MB/s eta 0:00:00


In [10]:
from botocore import UNSIGNED
from botocore.config import Config
import smart_open
import boto3

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))

def download_contents(blob_id, src_encoding):
    s3_url = f"s3://softwareheritage/content/{blob_id}"
    try:
        with smart_open.open(s3_url, "rb", compression=".gz", transport_params={"client": s3}) as fin:
            return fin.read().decode(src_encoding)
    except Exception as e:
        print(f"Error downloading blob {blob_id}: {e}")
        return ""

In [11]:
!pip install datasets
!pip install -U datasets fsspec huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.2/484.2 kB 41.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.31.1
    Uninstalling huggingface-hub-0.31.1:
      Successfully uninstalled huggingface-hub-0.31.1
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.

In [13]:
from datasets import load_dataset
from itertools import islice

ds_streamed = load_dataset("bigcode/the-stack-v2-dedup", "TypeScript", split="train", streaming=True)
ds = list(islice(ds_streamed, 30000))
print(f"Loaded dataset with {len(ds)} examples")

Resolving data files:   0%|          | 0/757 [00:00<?, ?it/s]

Loaded dataset with 30000 examples


In [14]:
def process_chunk(idx_and_chunk):
    idx, chunk = idx_and_chunk
    parser = make_parser()
    chunk_funs = set()
    for ex in chunk:
        try:
            blob_id = ex["blob_id"]
            encoding = ex.get("src_encoding", "utf-8")
            src = download_contents(blob_id, encoding)
            if not src.strip():
                continue
            buf = bytes(src, "utf8")
            tree = parser.parse(buf)
            captures = TS_FUNCTION_QUERY.captures(tree.root_node)
            chunk_funs.update([
                node_to_string(buf, node)
                for node, typ in captures if typ == "function.def"
            ])
        except:
            continue
    return chunk_funs

In [15]:
from multiprocessing import Pool
import os
import signal

# Initialize
funs = set()
NUMWORKERS = os.cpu_count()
PARSERS = [make_parser() for _ in range(NUMWORKERS)]
total_len = len(ds)
CHUNK_SIZE = 2000 * NUMWORKERS
#CHUNK_SIZE = total_len // 100

chunk = []
p = Pool(NUMWORKERS)

# Timeout handler
def timeout_handler(_, __):
    raise TimeoutError("Processing timed out")

# Main loop
for i, ex in enumerate(ds):
    # Progress marker every 1%
    #if i % (total_len // 100) == 0:
    if i % (2000 * NUMWORKERS) == 0:
        print(f"{i}/{total_len}")

    try:
        chunk.append(ex)

        if len(chunk) == CHUNK_SIZE or i == total_len - 1:
            chunk_idx = i // CHUNK_SIZE
            print(f"Processing chunk {chunk_idx}")
            print("Getting new functions...")

            # Divide chunk across workers
            subchunk_size = len(chunk) // NUMWORKERS
            subchunks = [chunk[j:j + subchunk_size] for j in range(0, len(chunk), subchunk_size)]

            try:
                new_funs_iter = p.imap(process_chunk, [(i, subchunk) for i, subchunk in enumerate(subchunks)])
                signal.signal(signal.SIGALRM, timeout_handler)
                signal.alarm(600)

                # Merge results
                while True:
                    try:
                        funs.update(next(new_funs_iter))
                    except StopIteration:
                        break

                signal.alarm(0)

                # Progress output
                num_functions = len(funs)
                progress_pct = (num_functions / total_len) * 100

                print(f"Done with chunk {chunk_idx}, total functions: {num_functions}")
                print(f"{num_functions}/{total_len} functions extracted so far ({progress_pct:.2f}%)")

            except TimeoutError:
                signal.alarm(0)
                print("Timeout. Restarting pool...")
                p.terminate()
                p.join()
                p = Pool(NUMWORKERS)

            chunk = []

    except Exception as e:
        # Handle per-example failure
        chunk = []

p.close()

0/30000
Processing chunk 0
Getting new functions...
Done with chunk 0, total functions: 1365
1365/30000 functions extracted so far (4.55%)
4000/30000
Processing chunk 1
Getting new functions...
Done with chunk 1, total functions: 2539
2539/30000 functions extracted so far (8.46%)
8000/30000
Processing chunk 2
Getting new functions...
Done with chunk 2, total functions: 3533
3533/30000 functions extracted so far (11.78%)
12000/30000
Processing chunk 3
Getting new functions...
Done with chunk 3, total functions: 4662
4662/30000 functions extracted so far (15.54%)
16000/30000
Processing chunk 4
Getting new functions...
Done with chunk 4, total functions: 5606
5606/30000 functions extracted so far (18.69%)
20000/30000
Processing chunk 5
Getting new functions...
Done with chunk 5, total functions: 6595
6595/30000 functions extracted so far (21.98%)
24000/30000
Processing chunk 6
Getting new functions...
Done with chunk 6, total functions: 7921
7921/30000 functions extracted so far (26.40%)


In [16]:
from datasets import Dataset

new_ds_dict = {
    "content": list(funs),
    "id": list(range(len(funs)))
}
new_ds = Dataset.from_dict(new_ds_dict)
new_ds.save_to_disk("/content/drive/MyDrive/Checkpoints/001")
print("content saved to /content/drive/MyDrive/Checkpoints/001")

Saving the dataset (0/1 shards):   0%|          | 0/8372 [00:00<?, ? examples/s]

content saved to /content/drive/MyDrive/Checkpoints/001


In [17]:
ds = new_ds

In [18]:
ds

Dataset({
    features: ['content', 'id'],
    num_rows: 8372
})

### SEED GATHERING HIGH-QUALITY SUBSET

In [8]:
from datasets import load_from_disk
ds = load_from_disk("/content/Functions")
print(f'Running functions {len(ds)} Successfully')

Running functions 8372 Successfully


In [9]:
!pip install tree_sitter==0.20.1

In [12]:
from tree_sitter import Language, Parser
LANGUAGE = Language("build/my-languages.so", "typescript")

In [13]:
import tree_sitter
print(tree_sitter.__file__)

/usr/local/lib/python3.11/dist-packages/tree_sitter/__init__.py


In [14]:
from tree_sitter import Language

# Load from the compiled .so file
lang = Language("build/my-languages.so", "typescript")
print(lang)

In [15]:
!cat /content/tree_sitter_parser.py

from tree_sitter import Language, Parser

# Load the compiled TypeScript language from the shared object file
LANGUAGE = Language("build/my-languages.so", "typescript")

# Global parser setup
global_parser = Parser()
global_parser.set_language(LANGUAGE)

# Query to capture function names
QUERY = LANGUAGE.query(
    """
(function_declaration name: (identifier) @fn-name)
"""
)

def get_fn_name(code, parser=global_parser):
    src = bytes(code, "utf8")
    tree = parser.parse(src)
    node = tree.root_node
    for cap, typ in QUERY.captures(node):
        if typ == "fn-name":
            return node_to_string(src, cap)
    return None

def node_to_string(src: bytes, node):
    return src[node.start_byte : node.end_byte].decode("utf8")

def make_parser():
    _parser = Parser()
    _parser.set_language(LANGUAGE)
    return _parser

RETURN_QUERY = LANGUAGE.query(
    """
(return_statement) @return
"""
)

def does_have_return(src, parser=global_parser):
    tree = parser.parse(bytes(src, "ut

In [16]:
correct_code = '''
from tree_sitter import Language, Parser

# Load the compiled TypeScript language from the shared object file
LANGUAGE = Language("build/my-languages.so", "typescript")

# Global parser setup
global_parser = Parser()
global_parser.set_language(LANGUAGE)

# Query to capture function names
QUERY = LANGUAGE.query(
    """
(function_declaration name: (identifier) @fn-name)
"""
)

def get_fn_name(code, parser=global_parser):
    src = bytes(code, "utf8")
    tree = parser.parse(src)
    node = tree.root_node
    for cap, typ in QUERY.captures(node):
        if typ == "fn-name":
            return node_to_string(src, cap)
    return None

def node_to_string(src: bytes, node):
    return src[node.start_byte : node.end_byte].decode("utf8")

def make_parser():
    _parser = Parser()
    _parser.set_language(LANGUAGE)
    return _parser

RETURN_QUERY = LANGUAGE.query(
    """
(return_statement) @return
"""
)

def does_have_return(src, parser=global_parser):
    tree = parser.parse(bytes(src, "utf8"))
    root = tree.root_node
    captures = RETURN_QUERY.captures(root)
    for node, _ in captures:
        if len(node.children) <= 1:
            continue
        else:
            return True
    return False

if __name__ == "__main__":
    code = """
function test(): string {
  return "hi";
}
"""
    print(global_parser.parse(bytes(code, "utf8")).root_node.sexp())
'''

with open("/content/tree_sitter_parser.py", "w") as f:
    f.write(correct_code.strip())

In [17]:
from tree_sitter import Language, Parser

# Load compiled TypeScript grammar from the .so file
LANGUAGE = Language("build/my-languages.so", "typescript")

# Create and configure the parser
def make_parser():
    parser = Parser()
    parser.set_language(LANGUAGE)
    return parser

# Global parser instance
global_parser = make_parser()

In [18]:
# TypeScript return statement query
RETURN_QUERY = LANGUAGE.query("""
(return_statement) @return
""")

In [19]:
def does_have_return(src, parser=None):
    if parser is None:
        parser = make_parser()
    try:
        tree = parser.parse(bytes(src, "utf8"))
        root = tree.root_node
        captures = RETURN_QUERY.captures(root)
        return len(captures) > 0
    except:
        return False

In [20]:
import subprocess
import tempfile
import signal
import hashlib
import os
import argparse
from typing import List, Dict
from tqdm import tqdm
from tree_sitter_parser import LANGUAGE, global_parser

RETURN_QUERY = LANGUAGE.query("""
(return_statement) @return
""")

def does_have_return(src):
    tree = global_parser.parse(bytes(src, "utf8"))
    root = tree.root_node
    captures = RETURN_QUERY.captures(root)
    for node, _ in captures:
        # if it doesn't have an argument, it's not a return with a value
        if len(node.children) <= 1:  # includes "return" itself
            continue
        else:
            return True
    return False

# runs mypy in the given directory, returns stdout
# then, it logs the number of errors for each file
def run_mypy(d):
    try:
        outs = subprocess.run(
            ["mypy", "."],
            cwd=d,
            capture_output=True,
            timeout=120,
            text=True,
        ).stdout
    except Exception as e:
        print(e)
        return None

    filemap = {}
    lines = outs.split("\n")
    for line in lines:
        if line.strip():
            parts = line.split(":")
            if len(parts) >= 2:
                file = parts[0].split("/")[-1]
                if file not in filemap:
                    filemap[file] = 0
                if "error:" in line:
                    filemap[file] += 1

    return filemap

def typecheck_batch(files: List[str]) -> Dict[str, str]:
    # Create a temporary directory using the tempfile module
    filemap: Dict[str, str] = {}
    with tempfile.TemporaryDirectory() as tempdir:
        for contents in files:
            hash_object = hashlib.sha1(bytes(contents, "utf8"))
            hex_dig = hash_object.hexdigest()
            filemap[hex_dig] = contents
            name = os.path.join(tempdir, hex_dig + ".py")
            with open(name, "w") as f:
                f.write(contents)

        # Run mypy in the temporary directory
        typecheck_map = run_mypy(tempdir)
        print(typecheck_map)

        if typecheck_map is None:
            return {}

        for contents, errors in typecheck_map.items():
            no_py = contents.replace(".py", "")
            if errors == 0:
                continue
            if no_py in filemap:
                del filemap[no_py]

        print(f"Pass rate: {len(filemap)}/{len(files)}")
        return filemap

def infer_imports(code: str) -> str:
    import autoimport
    try:
        def handler(signum, frame):
            raise Exception("Timeout")
        signal.signal(signal.SIGALRM, handler)
        signal.alarm(10)
        inferred = autoimport.fix_code(code)
        signal.alarm(0)
        return inferred
    except Exception as e:
        signal.alarm(0)
        print(f"Error while inferring imports: {e}")
        return code

In [21]:
print("Filtering to only functions with return statements")
ds = ds.filter(lambda ex: does_have_return(
    ex["content"]), num_proc=os.cpu_count())

Parameter 'function'=<function <lambda> at 0x793562deda80> of the transform datasets.arrow_dataset.Dataset.filter@2.0.1 couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Filtering to only functions with return statements


Filter (num_proc=2):   0%|          | 0/8372 [00:00<?, ? examples/s]

In [22]:
ds

Dataset({
    features: ['content', 'id'],
    num_rows: 6524
})

In [23]:
!pip install mypy

In [24]:
import datasets

In [25]:
# if args.infer_imports:
#     print("Inferring imports for functions")
#     ds = ds.map(lambda ex: {"content": infer_imports(
#         ex["content"])}, num_proc=os.cpu_count())

batch = []
max_i = len(ds) - 1

new_ds = {
    "content": [],
    "sha1": [],
    "id": [],
}

e_id = 0

for i, ex in enumerate(tqdm(ds, total=len(ds))):
    try:
        code = ex["content"]

        batch.append(code)

        if len(batch) == 250 or i == max_i:
            filemap = typecheck_batch(batch)
            for sha1, contents in filemap.items():
                new_ds["content"].append(contents)
                new_ds["sha1"].append(sha1)
                new_ds["id"].append(e_id)
                e_id += 1
            batch = []

    except Exception as e:
        print(f"There was an error: {e}")
        continue

new_ds_hf = datasets.Dataset.from_dict(new_ds)

  4%|▍         | 250/6524 [00:00<00:12, 501.57it/s]

{'0077fae4337fde7e03fb8b08b20a40826fdb449a.py': 1}
Pass rate: 249/250


  8%|▊         | 500/6524 [00:00<00:08, 676.99it/s]

{'002de561f9e0353ac6b17790c5de37fca59a45e0.py': 1}
Pass rate: 249/250


 11%|█▏        | 750/6524 [00:00<00:06, 836.65it/s]

...

{'016718e4c02dea667747896c5874a3671c7bb289.py': 1}
Pass rate: 249/250


100%|█████████▉| 6500/6524 [00:09<00:00, 477.65it/s]

{'009389ac971a12e3a7a4a7823d896d3e1d84faa1.py': 1}
Pass rate: 249/250


100%|██████████| 6524/6524 [00:10<00:00, 621.55it/s]

{'15bd7cd463acdb841978d8a53b9405fb3150d63a.py': 1}
Pass rate: 23/24


In [26]:
print(new_ds_hf['content'][0])

async function getAllUrl () {

    const mongoDbUrl = 'mongodb://' + dbUser + ':' + dbPassword + '@127.0.0.1:27017/';

    let connection = await MongoClient.connect(mongoDbUrl, {
      useNewUrlParser: true,
    });
    let db = await connection.db(dbName);
    const urls = db.collection('urls');

    const allUrls = await urls.find().toArray();

    await connection.close();
    
    return allUrls;
}


In [27]:
save_dir = "../datasets/seed2"

In [28]:
new_ds_hf.save_to_disk(save_dir)

Saving the dataset (0/1 shards):   0%|          | 0/6497 [00:00<?, ? examples/s]

### SEED GATHERING FILTER DATASET

In [71]:
from datasets import load_from_disk
ds = load_from_disk("/content/Functions")
print(f'Running functions {len(ds)} Successfully')

Running functions 8372 Successfully


In [72]:
!git clone https://github.com/kamaravichow/seed-gathering.git

Cloning into 'seed-gathering'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 28 (delta 9), reused 3 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 21.30 KiB | 21.30 MiB/s, done.
Resolving deltas: 100% (9/9), done.


In [73]:
!cp -r seed-gathering/* /content/
!rm -rf seed-gathering

In [74]:
!pip install -r requirements.txt

In [75]:
pip install torch==2.1.2+cu121 torchvision==0.16.2+cu121 torchaudio==2.1.2 --extra-index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 GB 365.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 26.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 103.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 MB 10.6 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.2.0
    Uninstalling triton-3.2.0:
      Successfully uninstalled triton-3.2.0
  Attempting uninstall: torch
    Found existing installation: torch 2.6.0+cu124
    Uninstalling torch-2.6.0+cu124:
      Successfully uninstalled torch-2.6.0+cu124
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.21.0+cu124
    Uninstalling torchvision-0.21.0+cu124:
      Successfully uninstalled torchvision-0.21.0+cu124
  Attempting uninstall: torchaudio
    Found existing installation: torchaudio 2.6.0+

In [76]:
!pip install torch

In [77]:
pip install --upgrade vllm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.2/253.2 MB 6.0 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 2.1.0
    Uninstalling triton-2.1.0:
      Successfully uninstalled triton-2.1.0
  Attempting uninstall: torch
    Found existing installation: torch 2.1.2+cu121
    Uninstalling torch-2.1.2+cu121:
      Successfully uninstalled torch-2.1.2+cu121
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.16.2+cu121
    Uninstalling torchvision-0.16.2+cu121:
      Successfully uninstalled torchvision-0.16.2+cu121
  Attempting uninstall: torchaudio
    Found existing installation: torchaudio 2.1.2+cu121
    Uninstalling torchaudio-2.1.2+cu121:
      Successfully uninstalled torchaudio-

In [78]:
pip install datasets

In [79]:
!pip install numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 108.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yfinance 0.2.59 requires protobuf<6,>=5.29.0, but you have protobuf 4.25.7 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
cuml-cu12 25.2.1 requires numba<0.61.0a0,>=0.59.1, but you have numba 0.61.2 which is incompatible.
distributed-ucxx-cu12 0.42.0 requires numba<0.61.0a0,>=0.59.1, but you have numba 0.61.2 which is incompatible.
dask-cuda 25.2.0 requires numba<0.61.0a0,>=0.59.1, but you have numba 0.61.2 which is incompatible.
ydf 0.11.0 requi

In [1]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

from vllm import LLM, SamplingParams
print("vLLM loaded successfully")

Torch version: 2.6.0+cu124
CUDA available: True
INFO 05-14 17:45:28 [__init__.py:239] Automatically detected platform cuda.
vLLM loaded successfully


In [29]:
correct_code = '''
from tree_sitter import Language, Parser

# Load the compiled TypeScript language from the shared object file
LANGUAGE = Language("build/my-languages.so", "typescript")

# Global parser setup
global_parser = Parser()
global_parser.set_language(LANGUAGE)

# Query to capture function names
QUERY = LANGUAGE.query(
    """
(function_declaration name: (identifier) @fn-name)
"""
)

def get_fn_name(code, parser=global_parser):
    src = bytes(code, "utf8")
    tree = parser.parse(src)
    node = tree.root_node
    for cap, typ in QUERY.captures(node):
        if typ == "fn-name":
            return node_to_string(src, cap)
    return None

def node_to_string(src: bytes, node):
    return src[node.start_byte : node.end_byte].decode("utf8")

def make_parser():
    _parser = Parser()
    _parser.set_language(LANGUAGE)
    return _parser

RETURN_QUERY = LANGUAGE.query(
    """
(return_statement) @return
"""
)

def does_have_return(src, parser=global_parser):
    tree = parser.parse(bytes(src, "utf8"))
    root = tree.root_node
    captures = RETURN_QUERY.captures(root)
    for node, _ in captures:
        if len(node.children) <= 1:
            continue
        else:
            return True
    return False

if __name__ == "__main__":
    code = """
function test(): string {
  return "hi";
}
"""
    print(global_parser.parse(bytes(code, "utf8")).root_node.sexp())
'''

with open("/content/tree_sitter_parser.py", "w") as f:
    f.write(correct_code.strip())

In [30]:
import datasets
import os
from tree_sitter_parser import global_parser, LANGUAGE, does_have_return, make_parser
import benchmark_data
from tqdm import tqdm
import torch
import argparse
from vllm import LLM, SamplingParams
import random

In [31]:
#FN_BLOCK_QUERY = LANGUAGE.query("""
#(function_definition
#  body: (block) @fn-block)
# """)
# Tree-sitter query for TypeScript
FN_BLOCK_QUERY = LANGUAGE.query("""
(function_declaration
  body: (statement_block) @fn-block)
""")

# Dummy docstring extractor for TypeScript
def ts_extract_docstring(code):
    lines = code.strip().splitlines()
    doc = ""
    if lines and lines[0].strip().startswith("//"):
        doc = lines[0].strip().lstrip("//").strip()
        code = "\n".join(lines[1:])
    return doc, code

# Template for a few-shot example
def template_few_shot(code, answer, rationale):
    doc, code = ts_extract_docstring(code)
    assert answer in ("Yes", "No")
    prompt = f"""<issue_start>username_0: I have a function in TypeScript and I'd like someone to check my description of this function.
I'm doing this so that I can write a good docstring for this function.

Here is the code for the function:
```ts
{code}
```

Here is my description of this program:
```
{doc}
```

Do not attempt to execute the function or to judge its correctness.
Answer with \"Yes\" or \"No\" depending on if my description has enough information alone to re-implement the function.
Also, answer with \"No\" if the description does not match the function.<issue_comment>username_1: Sure, no problem. I will be able to help.
My answer is: {answer}

{rationale}

Upvotes: 200"""
    return prompt

FEW_SHOTS = []

# Few-shot examples in TypeScript
FEW_SHOTS += [
    (
        """// Returns the maximum of two numbers
function min(a: number, b: number): number {
  return a < b ? a : b;
}""",
        "No",
        "The description says maximum, but the function returns the minimum."
    ),
    (
        """// Multiply two numbers
function multiply(x: number, y: number): number {
  return x * y;
}""",
        "Yes",
        "The description accurately reflects the multiplication logic."
    ),
    (
        """// Count number of vowels in a string
function countVowels(str: string): number {
  return str.split('').filter(c => 'aeiou'.includes(c)).length;
}""",
        "Yes",
        "The function and description align in purpose and logic."
    ),
    (
        """// Return true if number is even
function isOdd(n: number): boolean {
  return n % 2 === 0;
}""",
        "No",
        "The function checks for evenness, but the description says odd."
    ),
]


# Prompt formatter using the few-shots
def prompt_fmt(code):
    doc, code = ts_extract_docstring(code)
    random.shuffle(FEW_SHOTS)
    buf = ""
    for few in FEW_SHOTS:
        buf += template_few_shot(*few)
    buf += f"""<issue_start>username_0: I have a function in TypeScript and I'd like someone to check my description of this function.
I'm doing this so that I can write a good docstring for this function.

Here is the code for the function:
```ts
{code}
```

Here is my description of this program:
```
{doc}
```

Do not attempt to execute the function or to judge its correctness.
Answer with \"Yes\" or \"No\" depending on if my description has enough information alone to re-implement the function.
Also, answer with \"No\" if the description does not match the function.
Upvotes: 100<issue_comment>username_1: Sure, no problem. I will be able to help.
My answer is:"""
    return buf

# Auto-detect float16 / bfloat16
def auto_dtype():
    if torch.cuda.is_bf16_supported():
        return "bfloat16"
    return "auto"

# Chunkify function for batching prompts
def chunkify(lst, n):
    chunks = []
    for i in range(0, len(lst), n):
        chunk = []
        for j in range(n):
            if i + j < len(lst):
                chunk.append(lst[i + j])
        chunks.append(chunk)
    return chunks

In [32]:
dataset = new_ds_hf

In [33]:
dataset.save_to_disk("/content/drive/MyDrive/filterdataset")
print("content saved to /content/drive/MyDrive/filterdataset")

Saving the dataset (0/1 shards):   0%|          | 0/6497 [00:00<?, ? examples/s]

content saved to /content/drive/MyDrive/filterdataset


In [34]:
import shutil

shutil.make_archive('/content/filterdataset', 'zip', '/content/drive/MyDrive/filterdataset')

'/content/filterdataset.zip'

In [35]:
print(f"Loaded {len(dataset)} examples. Running pre-filtering...")

Loaded 6497 examples. Running pre-filtering...


In [36]:
for i in range(10):
    print(f"\n==== Sample {i} ====")
    print(dataset[i]['content'][:500])


==== Sample 0 ====
async function getAllUrl () {

    const mongoDbUrl = 'mongodb://' + dbUser + ':' + dbPassword + '@127.0.0.1:27017/';

    let connection = await MongoClient.connect(mongoDbUrl, {
      useNewUrlParser: true,
    });
    let db = await connection.db(dbName);
    const urls = db.collection('urls');

    const allUrls = await urls.find().toArray();

    await connection.close();
    
    return allUrls;
}

==== Sample 1 ====
function applySeparableKernel3x3(kernelX: number[], kernelY: number[], grayscale: Uint8Array, width: number, height: number) {

        let dst: Float32Array = new Float32Array(width * height);

        const padding = 1;
        let idx = padding * width + padding;
        for (let y: number = padding; y < height - padding; y++) {
            for (let x: number = padding; x < width - padding; x++) {

                let sum =
                    grayscale[idx - 1 - 0] * kernelX[0] +
           

==== Sample 2 ====
function get2d<T>(rows: number, 

In [37]:
print(f"Loaded {len(dataset)} examples. Running pre-filtering...")

BAD_WORDS = ["todo", "fixme", "bug"]
BAD_IMPORTS = ["argparse", "os", "subprocess", "sys", "setuptools",
               "distutils", "matplotlib", "seaborn"]
BAD_IMPORTS = [f"import {b}" for b in BAD_IMPORTS] + \
    [f"from {b}" for b in BAD_IMPORTS]
BAD_SUBSTRINGS = BAD_WORDS + BAD_IMPORTS

bench_filter = benchmark_data.filter_out()
all_bench = bench_filter["human_eval_docstrings"] + \
    bench_filter["human_eval_solutions"] + \
    bench_filter["mbpp_docstrings"] + \
    bench_filter["mbpp_solutions"]

Loaded 6497 examples. Running pre-filtering...


README.md:   0%|          | 0.00/9.06k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/33.9k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/60.9k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/14.0k [00:00<?, ?B/s]

...

Generating prompt split:   0%|          | 0/7 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

num strings from mbpp_docstrings: 120
num strings from mbpp_solutions: 120
num strings from human_eval_docstrings: 164
num strings from human_eval_solutions: 161


In [38]:
def pre_filtering(ex):
    code = ex["content"]
    code_bytes = code.encode("utf-8")

    # Filter out bad substrings (e.g., TODOs, bad imports)
    lower = code.lower()
    if any(word in lower for word in BAD_SUBSTRINGS):
        return False

    if any(b in code for b in all_bench):
        return False

    # Skip overly long functions
    lines = code.split("\n")
    if len(lines) > 150:
        return False

    # Skip Python-style no-arg functions (optional for TypeScript)
    if any(line.strip().startswith("function") and "():" in line for line in lines):
        return False

    # Must contain a return statement (modified to not use parser argument)
    if not does_have_return(code):  # Remove parser argument
        return False

    # Must have a parseable function body block
    try:
        tree = global_parser.parse(code_bytes)
        captures = FN_BLOCK_QUERY.captures(tree.root_node)
        if not captures:
            return False

        block, _ = captures[0]
        # Optional: look for docstring-like structure (rare in TS)
        first_child = block.children[0]
        if first_child.type == "expression_statement":
            doc_node = first_child.children[0]
            if doc_node.type == "string":
                docstring = doc_node.text.decode("utf-8")
                if docstring.startswith('"""') and docstring.endswith('"""'):
                    return True
        return True
    except Exception as e:
        print(f"Error in filtering: {e}")
        return False

# Make sure does_have_return is defined like this:
def does_have_return(code):
    """Check if the code contains a return statement"""
    return "return " in code

In [39]:
threads = os.cpu_count() - 1
dataset = dataset.filter(pre_filtering, num_proc=threads)

print(f"Filtered dataset contains {len(dataset)} high-quality functions")

Filter:   0%|          | 0/6497 [00:00<?, ? examples/s]

Filtered dataset contains 5791 high-quality functions


In [40]:
dataset

Dataset({
    features: ['content', 'sha1', 'id'],
    num_rows: 5791
})

In [41]:
dataset.save_to_disk("/content/drive/MyDrive/filterdataset_hqfun")
print("content saved to /content/drive/MyDrive/filterdataset_hqfun")

Saving the dataset (0/1 shards):   0%|          | 0/5791 [00:00<?, ? examples/s]

content saved to /content/drive/MyDrive/filterdataset_hqfun


In [42]:
import shutil

shutil.make_archive('/content/filterdataset_hqfun', 'zip', '/content/drive/MyDrive/filterdataset_hqfun')

'/content/filterdataset_hqfun.zip'

### SEED GATHERING LLM MODEL RUN

In [43]:
!apt-get install git-lfs
!pip install -q huggingface_hub

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.


In [44]:
!git lfs install
!git clone https://huggingface.co/bigcode/starcoder2-3b /content/StarCoder

Git LFS initialized.
Cloning into '/content/StarCoder'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 65 (delta 26), reused 0 (delta 0), pack-reused 3 (from 1)
Unpacking objects: 100% (65/65), 1.13 MiB | 1.68 MiB/s, done.
^C


In [45]:
!pip install -U --force-reinstall transformers

!pip uninstall -y torch torchvision

!pip install torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121

  Using cached huggingface_hub-0.31.2-py3-none-any.whl.metadata (13 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 52.5 MB/s eta 0:00:00
Using cached huggingface_hub-0.31.2-py3-none-any.whl (484 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 763.0/763.0 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 792.7/792.7 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 7.2 MB/s eta 

Found existing installation: torch 2.6.0
Uninstalling torch-2.6.0:
  Successfully uninstalled torch-2.6.0
Found existing installation: torchvision 0.21.0
Uninstalling torchvision-0.21.0:
  Successfully uninstalled torchvision-0.21.0
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download.pytorch.org/whl/cu121/torch-2.1.2%2Bcu121-cp311-cp311-linux_x86_64.whl (2200.7 MB)
  Using cached https://download.pytorch.org/whl/cu121/torchvision-0.16.2%2Bcu121-cp311-cp311-linux_x86_64.whl (6.8 MB)
  Using cached https://download.pytorch.org/whl/triton-2.1.0-0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (89.2 MB)
  Attempting uninstall: triton
    Found existing installation: triton 3.2.0
    Uninstalling triton-3.2.0:
      Successfully uninstalled triton-3.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.6.

In [46]:
pip install vllm

  Using cached torch-2.6.0-cp311-cp311-manylinux1_x86_64.whl.metadata (28 kB)
  Using cached torchvision-0.21.0-cp311-cp311-manylinux1_x86_64.whl.metadata (6.1 kB)
  Using cached triton-3.2.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.4 kB)
Using cached torch-2.6.0-cp311-cp311-manylinux1_x86_64.whl (766.7 MB)
Using cached torchvision-0.21.0-cp311-cp311-manylinux1_x86_64.whl (7.2 MB)
Using cached triton-3.2.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (253.2 MB)
  Attempting uninstall: triton
    Found existing installation: triton 2.1.0
    Uninstalling triton-2.1.0:
      Successfully uninstalled triton-2.1.0
  Attempting uninstall: torch
    Found existing installation: torch 2.1.2+cu121
    Uninstalling torch-2.1.2+cu121:
      Successfully uninstalled torch-2.1.2+cu121
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.16.2+cu121
    Uninstalling torchvision-0.16.2+cu121:
      Successfully uninstalled torc

In [47]:
!pip install --upgrade --force-reinstall numpy==1.26.4

  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.5
    Uninstalling numpy-2.2.5:
      Successfully uninstalled numpy-2.2.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.6.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2025.3.2 which is incompatible.
yfinance 0.2.59 requires protobuf<6,>=5.29.0, but you have protobuf 4.25.7 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
cuml-cu12 25.2.1 requires numba<0.61.0a0,>=0.59.1, but you have numba 0.61.2 which is incompatible.
distributed-ucxx-cu12 0.42.0 requires numba<0.61.0a0,>=0.59.1, but you have nu

In [1]:
import numpy as np
print(np.__version__)

1.26.4


In [ ]:
#model = LLM(f"/content/StarCoder", dtype=auto_dtype(),
#            gpu_memory_utilization=0.95, tensor_parallel_size=1)

In [2]:
from vllm import LLM, SamplingParams

model = LLM(
    "bigcode/starcoder2-3b",
    dtype="auto",
    gpu_memory_utilization=0.95,
    tensor_parallel_size=1,
    tokenizer="bigcode/starcoder2-3b",
    trust_remote_code=True
)

INFO 05-14 18:09:40 [__init__.py:239] Automatically detected platform cuda.
INFO 05-14 18:09:46 [config.py:2968] Downcasting torch.float32 to torch.float16.
INFO 05-14 18:10:00 [config.py:717] This model supports multiple tasks: {'reward', 'score', 'embed', 'classify', 'generate'}. Defaulting to 'generate'.
WARNING 05-14 18:10:00 [arg_utils.py:1658] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 05-14 18:10:00 [llm_engine.py:240] Initializing a V0 LLM engine (v0.8.5.post1) with config: model='bigcode/starcoder2-3b', speculative_config=None, tokenizer='bigcode/starcoder2-3b', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=16384, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  devic

model.safetensors:   0%|          | 0.00/12.1G [00:00<?, ?B/s]

INFO 05-14 18:12:59 [weight_utils.py:281] Time spent downloading weights for bigcode/starcoder2-3b: 176.245659 seconds
INFO 05-14 18:12:59 [weight_utils.py:315] No model.safetensors.index.json found in remote.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 05-14 18:13:47 [loader.py:458] Loading weights took 47.73 seconds
INFO 05-14 18:13:48 [model_runner.py:1140] Model loading took 5.6778 GiB and 224.738142 seconds
INFO 05-14 18:13:53 [worker.py:287] Memory profiling takes 5.30 seconds
INFO 05-14 18:13:53 [worker.py:287] the current vLLM instance can use total_gpu_memory (14.74GiB) x gpu_memory_utilization (0.95) = 14.00GiB
INFO 05-14 18:13:53 [worker.py:287] model weights take 5.68GiB; non_torch_memory takes 0.05GiB; PyTorch activation peak memory takes 1.04GiB; the rest of the memory reserved for KV Cache is 7.24GiB.
INFO 05-14 18:13:54 [executor_base.py:112] # cuda blocks: 15815, # CPU blocks: 8738
INFO 05-14 18:13:54 [executor_base.py:117] Maximum concurrency for 16384 tokens per request: 15.44x
INFO 05-14 18:14:00 [model_runner.py:1450] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the C

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

INFO 05-14 18:15:18 [model_runner.py:1592] Graph capturing finished in 78 secs, took 0.21 GiB
INFO 05-14 18:15:18 [llm_engine.py:437] init engine (profile, create kv cache, warmup model) took 90.46 seconds


In [3]:
tokenizer = model.get_tokenizer()

In [4]:
from datasets import load_from_disk
dataset = load_from_disk("/content/5791_HQ_Functions")

In [5]:
print(f"Now running stage 3 filtering on {len(dataset)} examples...")

Now running stage 3 filtering on 5791 examples...


In [6]:
from tree_sitter import Language, Parser
LANGUAGE = Language('build/my-languages.so', 'typescript')


In [7]:
#FN_BLOCK_QUERY = LANGUAGE.query("""
#(function_definition
#  body: (block) @fn-block)
# """)
# Tree-sitter query for TypeScript
FN_BLOCK_QUERY = LANGUAGE.query("""
(function_declaration
  body: (statement_block) @fn-block)
""")

# Dummy docstring extractor for TypeScript
def ts_extract_docstring(code):
    lines = code.strip().splitlines()
    doc = ""
    if lines and lines[0].strip().startswith("//"):
        doc = lines[0].strip().lstrip("//").strip()
        code = "\n".join(lines[1:])
    return doc, code

# Template for a few-shot example
def template_few_shot(code, answer, rationale):
    doc, code = ts_extract_docstring(code)
    assert answer in ("Yes", "No")
    prompt = f"""<issue_start>username_0: I have a function in TypeScript and I'd like someone to check my description of this function.
I'm doing this so that I can write a good docstring for this function.

Here is the code for the function:
```ts
{code}
```

Here is my description of this program:
```
{doc}
```

Do not attempt to execute the function or to judge its correctness.
Answer with \"Yes\" or \"No\" depending on if my description has enough information alone to re-implement the function.
Also, answer with \"No\" if the description does not match the function.<issue_comment>username_1: Sure, no problem. I will be able to help.
My answer is: {answer}

{rationale}

Upvotes: 200"""
    return prompt

FEW_SHOTS = []

# Few-shot examples in TypeScript
FEW_SHOTS += [
    (
        """// Returns the maximum of two numbers
function min(a: number, b: number): number {
  return a < b ? a : b;
}""",
        "No",
        "The description says maximum, but the function returns the minimum."
    ),
    (
        """// Multiply two numbers
function multiply(x: number, y: number): number {
  return x * y;
}""",
        "Yes",
        "The description accurately reflects the multiplication logic."
    ),
    (
        """// Count number of vowels in a string
function countVowels(str: string): number {
  return str.split('').filter(c => 'aeiou'.includes(c)).length;
}""",
        "Yes",
        "The function and description align in purpose and logic."
    ),
    (
        """// Return true if number is even
function isOdd(n: number): boolean {
  return n % 2 === 0;
}""",
        "No",
        "The function checks for evenness, but the description says odd."
    ),
]


# Prompt formatter using the few-shots
def prompt_fmt(code):
    doc, code = ts_extract_docstring(code)
    random.shuffle(FEW_SHOTS)
    buf = ""
    for few in FEW_SHOTS:
        buf += template_few_shot(*few)
    buf += f"""<issue_start>username_0: I have a function in TypeScript and I'd like someone to check my description of this function.
I'm doing this so that I can write a good docstring for this function.

Here is the code for the function:
```ts
{code}
```

Here is my description of this program:
```
{doc}
```

Do not attempt to execute the function or to judge its correctness.
Answer with \"Yes\" or \"No\" depending on if my description has enough information alone to re-implement the function.
Also, answer with \"No\" if the description does not match the function.
Upvotes: 100<issue_comment>username_1: Sure, no problem. I will be able to help.
My answer is:"""
    return buf

# Auto-detect float16 / bfloat16
def auto_dtype():
    if torch.cuda.is_bf16_supported():
        return "bfloat16"
    return "auto"

# Chunkify function for batching prompts
def chunkify(lst, n):
    chunks = []
    for i in range(0, len(lst), n):
        chunk = []
        for j in range(n):
            if i + j < len(lst):
                chunk.append(lst[i + j])
        chunks.append(chunk)
    return chunks

In [8]:
#import datasets
#import os
from tree_sitter_parser import global_parser, LANGUAGE, does_have_return, make_parser
#import benchmark_data
#from tqdm import tqdm
#import torch
#import argparse
#from vllm import LLM, SamplingParams
#import random

In [9]:
def unindent(s):
    lines = s.splitlines()
    non_blank_lines = [line for line in lines if line.strip()]
    min_indent = min(len(line) - len(line.lstrip())
                     for line in non_blank_lines) if non_blank_lines else 0
    unindented_lines = [line[min_indent:] if len(
        line) >= min_indent else line for line in lines]
    return '\n'.join(unindented_lines)


def py_extract_docstring(code):
    first_doc = code.find('"""')
    assert first_doc != -1
    first_doc = first_doc + 3
    second_doc = code[first_doc+1:].find('"""')
    assert second_doc != -1
    second_doc = second_doc + first_doc + 1
    doc = code[first_doc:second_doc]
    doc = unindent(doc).strip()
    code = code[:first_doc-3] + code[second_doc+3:]
    return doc, code

In [10]:
import random
from tqdm import tqdm

In [11]:
dummy = 'function dummy() {\n  // \n}'
dummy_prompt = prompt_fmt(dummy)
few_shot_toks = len(tokenizer.encode(
    dummy_prompt)) - len(tokenizer.encode(dummy))
print(f"Few-shot prompt has {few_shot_toks} tokens")

Few-shot prompt has 993 tokens


In [12]:
prompts = []
for ex in tqdm(dataset, total=len(dataset), desc="Generating prompts"):
    code = ex["content"]
    toks = len(tokenizer.encode(code)) + few_shot_toks
    if toks > 16380:
        print(f"Skipping example with {toks} tokens")
        # to skip, just add dummy prompt
        prompts.append(dummy_prompt)
        continue
    p = prompt_fmt(code)
    prompts.append(p)

responses = []
for chunk in tqdm(chunkify(prompts, 512), desc="Generating responses"):
    outs = model.generate(chunk, SamplingParams(
        temperature=0.0, stop="\n", max_tokens=5))
    contents = [o.outputs[0].text for o in outs]
    for c in contents:
        yes_count = c.lower().count("yes")
        no_count = c.lower().count("no")
        if yes_count > no_count:
            responses.append(True)
        elif yes_count < no_count:
            responses.append(False)
        else:
            # default to No
            responses.append(False)

Generating responses:   0%|          | 0/12 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/512 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generating responses:   8%|▊         | 1/12 [03:19<36:38, 199.88s/it]

Processed prompts:   0%|          | 0/512 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generating responses:  17%|█▋        | 2/12 [06:51<34:26, 206.65s/it]

...

Generating responses:  83%|████████▎ | 10/12 [34:04<06:47, 203.84s/it]

Processed prompts:   0%|          | 0/512 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generating responses:  92%|█████████▏| 11/12 [37:21<03:21, 201.95s/it]

Processed prompts:   0%|          | 0/159 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generating responses: 100%|██████████| 12/12 [38:26<00:00, 192.18s/it]


In [13]:
dataset

Dataset({
    features: ['content', 'sha1', 'id'],
    num_rows: 5791
})

In [15]:
dataset.save_to_disk("/content/drive/MyDrive/newds_seed3")
print("content saved to /content/drive/MyDrive/newds_seed3")

Saving the dataset (0/1 shards):   0%|          | 0/5791 [00:00<?, ? examples/s]

content saved to /content/drive/MyDrive/newds_seed3


In [16]:
import shutil
shutil.make_archive('/content/newds_seed3', 'zip', '/content/drive/MyDrive/newds_seed3')

'/content/newds_seed3.zip'

In [17]:
subset = dataset.select(range(5000))

In [18]:
subset

Dataset({
    features: ['content', 'sha1', 'id'],
    num_rows: 5000
})

In [19]:
new_ds = subset.filter(  # horrible hack!
    lambda ex, i: responses[i] and "def dummy()" not in ex["content"], with_indices=True)
print(f"Filtered {len(dataset) - len(new_ds)} examples")

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Filtered 1149 examples


In [20]:
new_ds.save_to_disk("../datasets/seed3.1")

Saving the dataset (0/1 shards):   0%|          | 0/4642 [00:00<?, ? examples/s]

In [21]:
new_ds

Dataset({
    features: ['content', 'sha1', 'id'],
    num_rows: 4642
})

In [22]:
new_ds.save_to_disk("/content/drive/MyDrive/newds_seed3.1")
print("content saved to /content/drive/MyDrive/newds_seed3.1")

Saving the dataset (0/1 shards):   0%|          | 0/4642 [00:00<?, ? examples/s]

content saved to /content/drive/MyDrive/newds_seed3.1


In [23]:
import shutil

shutil.make_archive('/content/newds_seed3.1', 'zip', '/content/drive/MyDrive/newds_seed3.1')

'/content/newds_seed3.1.zip'